# From a public germline variant to its literature graph
Use Folklore for structured variant evidence, then Noodle for source-linked literature discovery. No secrets are required. Do not send patient, phenotype, family or private case data.

In [ ]:
import json
import urllib.request

PROTOCOL = "2026-07-28"


def call_tool(endpoint, name, arguments):
    body = {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "tools/call",
        "params": {
            "name": name,
            "arguments": arguments,
            "_meta": {
                "io.modelcontextprotocol/protocolVersion": PROTOCOL,
                "io.modelcontextprotocol/clientCapabilities": {},
            },
        },
    }
    request = urllib.request.Request(
        endpoint,
        data=json.dumps(body).encode(),
        headers={
            "Accept": "application/json",
            "Content-Type": "application/json",
            "MCP-Protocol-Version": PROTOCOL,
            "Mcp-Method": "tools/call",
            "Mcp-Name": name,
            "User-Agent": "notebook-folklore-noodle/0.1.0",
        },
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        document = json.load(response)
    if "error" in document or document.get("result", {}).get("isError"):
        raise RuntimeError("The service returned a bounded tool error")
    result = document["result"]
    return result.get("structuredContent", result)


FOLKLORE = "https://api.helena.bio/folklore/v1/mcp"
NOODLE = "https://api.helena.bio/noodle/v1/mcp"

In [ ]:
variant = call_tool(
    FOLKLORE,
    "search_variant_evidence",
    {"assembly": "GRCh38", "query": "chr17:43124028 CTC>C"},
)
interpretation = variant["result"]["interpretation"]
gene = interpretation["annotation"]["gene_symbol"]
print(gene, interpretation["evidence"]["clinvar_significance"])

In [ ]:
papers = call_tool(
    NOODLE,
    "search_biomedical_literature",
    {
        "query": f"{gene} homologous recombination germline variant",
        "limit": 5,
        "sort": "relevance",
    },
)
[(item.get("pmid"), item.get("title")) for item in papers.get("results", [])]

In [ ]:
seed_pmid = papers["results"][0]["pmid"]
graph = call_tool(NOODLE, "get_publication_neighborhood", {"pmid": seed_pmid})
print(
    graph.get("graph_version"), len(graph.get("nodes", [])), len(graph.get("edges", []))
)

Folklore results require qualified professional review and are not a diagnosis or treatment recommendation. Noodle ranking and graph relationships are discovery aids; preserve citations, edge reasons and provenance.